# 02 — Ülke/yer alanı boş kurumsal PSC sahipleri

EDA'da (01) aktif kurumsal PSC kayıtlarının bir kısmında `country_registered` ve `place_registered`
alanlarının boş olduğunu, bunların **48,407** tanesinin `looks_uk_registered()` tarafından
İngiliz olarak tanınmadığını görmüştük. Bu notebook şu soruyu cevaplar:

> Bu kayıtların ne kadarı aslında Companies House'ta kayıtlı bir İngiliz şirketi?

**Önemli bulgu:** Bu kayıtların `identification` bloğunda yalnızca `legal_authority` ve `legal_form`
var; `registration_number` neredeyse hiç dolu değil (48,407 kayıttan 1'i). Bu yüzden numara üzerinden
eşleştirme yapılamıyor; doğrulama **sahip adı ↔ Companies House ünvanı** eşleşmesiyle yapılıyor.

In [17]:
import difflib
import re

import pandas as pd

from kyb.config import Paths
from kyb.data.readers import iter_csv, iter_psc
from kyb.data.subset import CH_COLS, CORPORATE
from kyb.normalize import looks_uk_registered, normalize_company_number

psc_dosyalari = sorted((Paths().raw / "ch_psc").glob("*.zip"))
sirket_dosyalari = sorted((Paths().raw / "ch_basic").glob("*.zip"))
print("PSC:", [p.name for p in psc_dosyalari])
print("Şirket verisi:", [p.name for p in sirket_dosyalari])

PSC: ['persons-with-significant-control-snapshot-2026-11-21.zip']
Şirket verisi: ['BasicCompanyDataAsOneFile-2026-11-01.zip']


## 1. Ülke/yer alanı boş aktif kurumsal kayıtları topla

In [18]:
kayitlar = []
for _sirket_no, veri in iter_psc(psc_dosyalari):
    if veri.get("kind") not in CORPORATE or veri.get("ceased_on"):
        continue
    kimlik = veri.get("identification") or {}
    if not kimlik:
        continue
    if kimlik.get("country_registered") or kimlik.get("place_registered"):
        continue
    ham = kimlik.get("registration_number")
    kayitlar.append({
        "sahip_adi": veri.get("name") or "",
        "legal_authority": kimlik.get("legal_authority") or "",
        "legal_form": kimlik.get("legal_form") or "",
        "ham_numara": ham,
        "numara": normalize_company_number(ham),
        "uk_ipucu": looks_uk_registered(kimlik),
    })

df = pd.DataFrame(kayitlar)
print(f"Ülke/yer alanı boş aktif kurumsal kayıt: {len(df):,}")
print(f"Bunlardan looks_uk_registered=False olan: {(~df['uk_ipucu']).sum():,}  (EDA'da 48,407 bulmuştuk)")

Ülke/yer alanı boş aktif kurumsal kayıt: 58,152
Bunlardan looks_uk_registered=False olan: 48,407  (EDA'da 48,407 bulmuştuk)


## 2. Hangi alanlar dolu?

In [19]:
bos = df[~df["uk_ipucu"]]
print("Kayıt:", len(bos))
print("Ham numarası dolu olan:", bos["ham_numara"].fillna("").astype(str).str.strip().ne("").sum())
print("Normalize edilebilen:", bos["numara"].notna().sum())
print("\nEn sık ham numaralar:")
print(bos["ham_numara"].dropna().astype(str).value_counts().head(15))
print("\nlegal_authority + legal_form doluluk:")
print("legal_authority dolu:", bos["legal_authority"].str.strip().ne("").sum())
print("legal_form dolu:", bos["legal_form"].str.strip().ne("").sum())

Kayıt: 48407
Ham numarası dolu olan: 1
Normalize edilebilen: 0

En sık ham numaralar:
ham_numara
14955535 (Company) 1207394 (Charity)    1
Name: count, dtype: int64

legal_authority + legal_form doluluk:
legal_authority dolu: 48407
legal_form dolu: 48407


In [20]:
def ust_n(seri, n=20):
    return seri.fillna("").str.strip().str.upper().value_counts().head(n)

print(f"Numarası normalize edilebilenler: %{bos['numara'].notna().mean() * 100:.1f}")
display(ust_n(bos["legal_authority"]).to_frame("adet"))
display(ust_n(bos["legal_form"]).to_frame("adet"))

Numarası normalize edilebilenler: %0.0


,adet
legal_authority,
COMPANIES ACT 2006,24106
COMPANIES ACT,4309
THE COMPANIES ACT 2006,1248
LIMITED LIABILITY PARTNERSHIPS ACT 2000,824
ENGLISH,594
ENGLISH LAW,493
LIMITED COMPANY,470
COMPANIES ACT 1985,411
COMPANIES ACTS,320


,adet
legal_form,
LIMITED COMPANY,19344
PRIVATE LIMITED COMPANY,8962
LIMITED,2274
PRIVATE COMPANY LIMITED BY SHARES,1781
LIMITED LIABILITY COMPANY,1003
LTD,990
LIMITED PARTNERSHIP,777
TRUST,701
PUBLIC LIMITED COMPANY,613


## 3. Companies House listesiyle eşleştirme

Numara olmadığı için iki yol birlikte kullanılıyor:

1. **Numara ile** (numarası olan nadir kayıtlar için) → ünvan benzerliği ile kontrol edilir.
2. **Ünvan ile** — sahip adı ve CH ünvanı aynı kurala göre normalize edilip (büyük harf, noktalama
   silme, `LTD → LIMITED`, `PLC → PUBLIC LIMITED COMPANY`) birebir aranır.

In [21]:
KISALTMALAR = {"LTD": "LIMITED", "PLC": "PUBLIC LIMITED COMPANY", "CO": "COMPANY", "LLP": "LIMITED LIABILITY PARTNERSHIP"}

def unvan_normalize(metin):
    metin = (metin or "").upper().replace("&", " AND ")
    metin = re.sub(r"[^A-Z0-9 ]", " ", metin)
    return " ".join(KISALTMALAR.get(k, k) for k in metin.split())

def benzerlik(a, b):
    if not isinstance(b, str) or not b:
        return float("nan")
    return difflib.SequenceMatcher(None, unvan_normalize(a), unvan_normalize(b)).ratio()

df["ad_norm"] = df["sahip_adi"].map(unvan_normalize)

In [22]:
aranan_no = set(df["numara"].dropna())
aranan_ad = set(df["ad_norm"]) - {""}

unvanlar = {}        # numara -> CH ünvanı
isimden = {}         # normalize ünvan -> (numara, CH ünvanı)
for satir in iter_csv(sirket_dosyalari):
    ad = satir.get(CH_COLS["name"], "")
    no = normalize_company_number(satir.get(CH_COLS["number"]))
    if no in aranan_no:
        unvanlar[no] = ad
    ad_n = unvan_normalize(ad)
    if ad_n in aranan_ad and ad_n not in isimden:
        isimden[ad_n] = (no, ad)

df["ch_unvan"] = df["numara"].map(unvanlar)
df["ch_no_isimden"] = df["ad_norm"].map(lambda a: isimden.get(a, (None, None))[0])
df["ch_unvan_isimden"] = df["ad_norm"].map(lambda a: isimden.get(a, (None, None))[1])

print(f"Numara ile → aranan: {len(aranan_no):,} · bulunan: {len(unvanlar):,}")
print(f"Ünvan ile  → aranan: {len(aranan_ad):,} · bulunan: {len(isimden):,}")
print(f"Ünvanı CH'de bulunan kayıt (boş-ülke grubu): {df.loc[~df['uk_ipucu'], 'ch_no_isimden'].notna().sum():,}")

Numara ile → aranan: 0 · bulunan: 0
Ünvan ile  → aranan: 40,473 · bulunan: 24,160
Ünvanı CH'de bulunan kayıt (boş-ülke grubu): 29,721


In [23]:
# Numara eşleşmesi için ünvan benzerliği (numarası olmayanlar NaN kalır)
df["benzerlik"] = [benzerlik(a, b) for a, b in zip(df["sahip_adi"], df["ch_unvan"])]
df["benzerlik"] = df["benzerlik"].astype(float)
df["benzerlik"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: benzerlik, dtype: float64

In [24]:
eslesen = df[df["benzerlik"].notna()]
if eslesen.empty:
    print("Numara ile eşleşen kayıt yok — benzerlik dağılımı hesaplanamıyor.")
else:
    df["benzerlik_araligi"] = pd.cut(df["benzerlik"], bins=[0, 0.5, 0.7, 0.85, 0.95, 1.0], include_lowest=True)
    display(df["benzerlik_araligi"].value_counts().sort_index().to_frame("adet"))

orta = df[df["benzerlik"].between(0.5, 0.85)]
display(orta.sample(min(20, len(orta)), random_state=42)[["sahip_adi", "ch_unvan", "benzerlik"]])

Numara ile eşleşen kayıt yok — benzerlik dağılımı hesaplanamıyor.


,sahip_adi,ch_unvan,benzerlik


## 4. Kategorilere ayırma

In [25]:
ESIK = 0.85   # numara eşleşmelerinde ünvan benzerlik eşiği

def kategori(satir):
    if pd.notna(satir["numara"]):
        if pd.isna(satir["ch_unvan"]):
            return "C. Listede yok"
        if satir["benzerlik"] >= ESIK:
            return "A. Doğrulanmış İngiliz"
        return "B. Çakışma şüphesi"
    if pd.notna(satir["ch_no_isimden"]):
        return "A. Doğrulanmış İngiliz"
    return "D. Karar verilemez"

df["kategori"] = df.apply(kategori, axis=1)

ozet = df[~df["uk_ipucu"]]["kategori"].value_counts().sort_index().to_frame("adet")
ozet["yuzde"] = (ozet["adet"] / ozet["adet"].sum() * 100).round(1)
ozet

,adet,yuzde
kategori,,
A. Doğrulanmış İngiliz,29721,61.4
D. Karar verilemez,18686,38.6


## 5. Basit kural ne kadar iyi?

Numara olmadığı için kural yalnızca `legal_authority` metnine bakar. Doğruluk etiketi: ünvanı CH'de bulunanlar
(A) pozitif, bulunamayanlar (D) negatif kabul edilir. D'de İngiliz olup ünvanı farklı yazılmış şirketler
olabileceği için kesinlik değeri **alt sınır** olarak okunmalı.

In [26]:
KURAL_DESENI = r"COMPANIES ACT|COMPANY ACT|\bUK\b|UNITED KINGDOM|ENGLAND|ENGLISH|WALES|SCOTLAND|BRITISH|LIMITED LIABILITY PARTNERSHIPS? ACT"

df["kural"] = df["legal_authority"].str.upper().str.contains(KURAL_DESENI, regex=True)

etiketli = df[~df["uk_ipucu"]]
pozitif = etiketli["kategori"] == "A. Doğrulanmış İngiliz"
dogru_pozitif = (etiketli["kural"] & pozitif).sum()
yanlis_pozitif = (etiketli["kural"] & ~pozitif).sum()
kacirilan = (~etiketli["kural"] & pozitif).sum()

def oran(pay, payda):
    return pay / payda * 100 if payda else float("nan")

print(f"Kuralın yakaladığı: {etiketli['kural'].sum():,} / {len(etiketli):,}")
print(f"Kesinlik: %{oran(dogru_pozitif, dogru_pozitif + yanlis_pozitif):.1f} · "
      f"Duyarlılık: %{oran(dogru_pozitif, dogru_pozitif + kacirilan):.1f}")

Kuralın yakaladığı: 35,645 / 48,407
Kesinlik: %75.6 · Duyarlılık: %90.7
